### *Jorge Edgardo Viera*


## Módulo 4 · Tarea 3 — Bases de datos SQLite en Python

### ***Ejercicio 1. Creación de la base de datos y tabla de departamentos***

**NOTA**: Al usar *PRIMARY KEY AUNTOINCREMENT*  llevamos un registro interno del último valor autoincremental usado para cada tabla, y lo guardamos en una tabla que se crea paralela llamada **sqlite_sequence**

In [1]:
# Creamos la base de datos

import sqlite3
from pathlib import Path # Para manejar rutas de archivos de manera más sencilla

DB_PATH = Path("empresa.db")

conn = sqlite3.connect(DB_PATH) # Crea una conexión a la base de datos. Si el archivo no existe, se creará automáticamente.
conn.close() # Cerramos la conexión a la base de datos después de crearla

print(f"Base de datos creada en: {DB_PATH.resolve()}") # Imprime la ruta absoluta de la base de datos creada


# --------------Creamos la tabla de departamentos----------------

with sqlite3.connect(DB_PATH) as conn: # Implemetamos *with* para manejar la conexión a la base de datos de manera segura, asegurando que se cierre automáticamente
    cursor = conn.cursor() # Creamos un cursor para ejecutar comandos SQL
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS departamentos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre TEXT NOT NULL,
            presupuesto INTEGER NOT NULL
        )
    """) # Ejecutamos un comando SQL para crear la tabla de departamentos si no existe


print("Tabla 'departamentos' creada correctamente.") # Imprime un mensaje indicando que la tabla se ha creado

Base de datos creada en: C:\Users\DELL\Desktop\Tareas\empresa.db
Tabla 'departamentos' creada correctamente.


*Insertamos 5 departamentos con nombres y presupuestos diferentes.*

In [2]:
# Insertar departamentos
# Definimos una lista de tuplas, donde cada tupla representa un departamento con su nombre y presupuesto.
#Esta estructura es conveniente para usar con el método executemany.
departamentos = [
    ("Recursos Humanos", 50000),
    ("Marketing",        75000),
    ("Desarrollo",      100000),
    ("Ventas",           60000),
    ("Finanzas",         80000),
]

with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()
    # Utilizamos executemany para insertar múltiples registros de una sola vez, lo que es más eficiente que ejecutar un comando de inserción por cada departamento
    cursor.executemany("""
        INSERT INTO departamentos (nombre, presupuesto)
        VALUES (?, ?)
    """, departamentos)

print(f"{len(departamentos)} departamentos insertados correctamente.")

5 departamentos insertados correctamente.


### ***Ejercicio 2. Añadir empleados con detalles***

Acción importante en el código:  *Activamos la foreign key (CLAVE FORANEA) implementando **PRAGMA***

In [3]:
# --------------Creamos la tabla de empleados----------------

with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON") # Habilitamos el soporte para claves foráneas en SQLite, 
    # lo que es necesario para garantizar la integridad referencial entre las tablas de empleados y departamentos

    # Crear tabla de empleados con una clave foránea que referencia a la tabla de departamentos
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS empleados(
    id INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    puesto TEXT NOT NULL,
    salario REAL NOT NULL,
    fecha_contratacion DATE NOT NULL,
    departamento_id INTEGER NOT NULL,
    FOREIGN KEY (departamento_id) REFERENCES departamentos(id)
)
    """) # Ejecutamos un comando SQL para crear la tabla de empleados si no existe

print("Tabla 'empleados' creada correctamente.") # Imprime un mensaje indicando que la tabla se ha creado

# Insertamos 10 empleados con distintos puestos, salarios y fechas de contratación. y su relacion con los departamentos mediante el campo departamento_id. 
empleados = [
    ("Juan Pérez", "Gerente de Recursos Humanos", 60000, "2020-01-15", 1),
    ("María López", "Especialista en Marketing", 55000, "2021-03-22", 2),
    ("Carlos García", "Desarrollador Senior", 90000, "2019-07-30", 3),
    ("Ana Martínez", "Vendedora", 45000, "2022-05-10", 4),
    ("Luis Fernández", "Analista Financiero", 70000, "2020-11-05", 5),
    ("Sofía Torres", "Diseñadora Gráfica", 50000, "2021-08-18", 2),
    ("Miguel Sánchez", "Desarrollador Junior", 40000, "2022-02-14", 3),
    ("Laura Ramírez", "Asistente de Ventas", 42000, "2021-12-01", 4),
    ("Diego Morales", "Contador", 65000, "2020-09-25", 5),
    ("Elena Jiménez", "Especialista en Recursos Humanos", 58000, "2021-06-30", 1),
]

# Los insertamos a la base de datos utilizando executemany para eficiencia
with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()
    cursor.executemany("""
        INSERT INTO empleados (nombre, puesto, salario, fecha_contratacion, departamento_id)
        VALUES (?, ?, ?, ?, ?)
    """, empleados)

print(f"{len(empleados)} empleados insertados correctamente.") # Imprime un mensaje indicando que los empleados se han insertado correctamente

Tabla 'empleados' creada correctamente.
10 empleados insertados correctamente.


### ***Ejercicio 3. Consultas básicas con JOIN***

In [4]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row # Configuramos la conexión para que devuelva filas como objetos de tipo Row, lo que permite acceder a los campos por nombre

    # Consulta para obtener el nombre del empleado, puesto y el nombre del departamento al que pertenece
    print("=== Empleados y sus departamentos ===")
    consulta = """
        SELECT 
            e.nombre AS empleado, 
            e.puesto, 
            d.nombre AS departamento
        FROM empleados e
        JOIN departamentos d ON e.departamento_id = d.id
    """
    filas = conn.execute(consulta).fetchall() # Ejecutamos la consulta y obtenemos todas las filas resultantes

    # Imprimimos los resultados de la consulta
    for fila in filas:
        print(dict(fila)) # Convertimos cada fila a un diccionario para una visualización más clara de los datos

    # Realizamos una consulta que muestre únicamente los empleados con salario superior al salario promedio de todos los empleados.
    print("\n=== Empleados con salario superior al promedio ===")
    consulta_2 = """
        SELECT 
            nombre, 
            puesto, 
            salario
        FROM empleados
        WHERE salario > (SELECT AVG(salario) FROM empleados)
        ORDER BY salario DESC
    """
    filas = conn.execute(consulta_2).fetchall() # Implementamos fetchall para obtener todas las filas resultantes de la consulta
    for fila in filas:
        print(dict(fila)) # Convertimos cada fila a un diccionario para una visualización más clara de los datos

=== Empleados y sus departamentos ===
{'empleado': 'Juan Pérez', 'puesto': 'Gerente de Recursos Humanos', 'departamento': 'Recursos Humanos'}
{'empleado': 'María López', 'puesto': 'Especialista en Marketing', 'departamento': 'Marketing'}
{'empleado': 'Carlos García', 'puesto': 'Desarrollador Senior', 'departamento': 'Desarrollo'}
{'empleado': 'Ana Martínez', 'puesto': 'Vendedora', 'departamento': 'Ventas'}
{'empleado': 'Luis Fernández', 'puesto': 'Analista Financiero', 'departamento': 'Finanzas'}
{'empleado': 'Sofía Torres', 'puesto': 'Diseñadora Gráfica', 'departamento': 'Marketing'}
{'empleado': 'Miguel Sánchez', 'puesto': 'Desarrollador Junior', 'departamento': 'Desarrollo'}
{'empleado': 'Laura Ramírez', 'puesto': 'Asistente de Ventas', 'departamento': 'Ventas'}
{'empleado': 'Diego Morales', 'puesto': 'Contador', 'departamento': 'Finanzas'}
{'empleado': 'Elena Jiménez', 'puesto': 'Especialista en Recursos Humanos', 'departamento': 'Recursos Humanos'}

=== Empleados con salario super

### ***Ejercicio 4. Actualización de datos***

In [5]:
# Incrementamos un 15% el salario de los empleados contratados antes de 2020

with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()
    cursor.execute("""
        UPDATE empleados
        SET salario = salario * 1.15
        WHERE fecha_contratacion < '2020-01-01'
    """) # Ejecutamos un comando SQL para actualizar el salario de los empleados contratados antes de 2020, incrementándolo en un 15% 
    # Con los datos actuales, solo Carlos García fue contratado antes de 2020 (2019-07-30), así que su salario fue incrementado
    print(f"Empleados actualizados: {cursor.rowcount}")


    # Reducimos en un 10% el presupuesto de los departamentos cuyo gasto en salarios supere su presupuesto actual.
    
    cursor.execute("""
        UPDATE departamentos
        SET presupuesto = presupuesto * 0.90
        WHERE (
            SELECT SUM(e.salario)
            FROM empleados e
            WHERE e.departamento_id = departamentos.id
        ) > presupuesto
    """)
    # Los 5 departamentos se reducen porque todos tienen más gasto en salarios que presupuesto.
    print(f"Departamentos actualizados: {cursor.rowcount}")



Empleados actualizados: 1
Departamentos actualizados: 5


### ***Ejercicio 5. Eliminación y depuración de registros***

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()
    # Eliminamos a los empleados cuyo salario sea inferior a 1200.
    cursor.execute(
        "DELETE FROM empleados WHERE salario < ?;", #Implementacion de la sentencia DELETE FROM para eliminar datos, usado SIEMPRE el filtro WHERE
        (1200,)  # ← tupla con el valor numérico
    )
    print(f"Empleados eliminados: {cursor.rowcount}")
    # Realizamos una consulta que muestre cuántos empleados quedan en cada departamento.
    filas = conn.execute("""
        SELECT d.nombre, COUNT(e.id) AS total
        FROM departamentos d
        LEFT JOIN empleados e ON e.departamento_id = d.id
        GROUP BY d.id
    """).fetchall()
    print("\n==========EMPLEADOS POR DEPARTAMENTO==========")
    for fila in filas:
        print(f"{fila[0]}: {fila[1]} empleados")


Empleados eliminados: 0

==========EMPLEADOS POR DEPARTAMENTO==========
Recursos Humanos: 2 empleados
Marketing: 2 empleados
Desarrollo: 2 empleados
Ventas: 2 empleados
Finanzas: 2 empleados


### ***Ejercicio 6. Creación de la tabla de proyectos***

In [ ]:
# --------------Creamos la tabla de proyectos---------------- El ejercicio no solicita NOT NULL

with sqlite3.connect(DB_PATH) as conn: # Implemetamos *with* para manejar la conexión a la base de datos de manera segura, asegurando que se cierre automáticamente
    cursor = conn.cursor() # Creamos un cursor para ejecutar comandos SQL
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS proyectos (
            id              INTEGER PRIMARY KEY,
            nombre          TEXT,
            inicio          DATE,
            fin             DATE,
            departamento_id INTEGER,
            presupuesto     INTEGER
    )
""") # Ejecutamos un comando SQL para crear la tabla de proyectos si no existe


print("Tabla 'proyectos' creada correctamente.") # Imprime un mensaje indicando que la tabla se ha creado

# Insertamos 5 proyectos distribuidos en distintos departamentos con el uso de executemany.
proyectos = [
    ("Sistema ERP",       "2023-01-10", "2023-12-31", 1, 30000),
    ("Campaña Digital",   "2023-03-15", "2023-09-30", 2, 20000),
    ("App Móvil",         "2023-02-01", "2024-01-31", 3, 50000),
    ("Expansión Mercado", "2023-05-01", "2023-11-30", 4, 25000),
    ("Auditoría Anual",   "2023-04-01", "2023-07-31", 5, 15000),
]

with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()
    cursor.executemany("""
        INSERT INTO proyectos (nombre, inicio, fin, departamento_id, presupuesto)
        VALUES (?, ?, ?, ?, ?)
    """, proyectos)

print(f"{len(proyectos)} proyectos insertados correctamente.")


Tabla 'proyectos' creada correctamente.
5 proyectos insertados correctamente.


### ***Ejercicio 7. Relación entre empleados y proyectos***

In [ ]:
# --------------Creamos la tabla empleados_proyectos---------------- El ejercicio no solicita NOT NULL

with sqlite3.connect(DB_PATH) as conn: # Implemetamos *with* para manejar la conexión a la base de datos de manera segura, asegurando que se cierre automáticamente
    cursor = conn.cursor() # Creamos un cursor para ejecutar comandos SQL
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS empleados_proyectos (
            empleado_id     INTEGER,
            proyecto_id     INTEGER,
            horas_asignadas INTEGER,
            FOREIGN KEY (empleado_id) REFERENCES empleados(id),
            FOREIGN KEY (proyecto_id) REFERENCES proyectos(id)
        )
    """) # Ejecutamos un comando SQL para crear la tabla de empleados_proyectos si no existe

print("Tabla 'empleados_proyectos' creada correctamente.") # Imprime un mensaje indicando que la tabla se ha creado

# Generamos la variable relaciones usando los id ya registrados en las otras tablas y escribiendolos dentro de una tupla
relaciones = [
    (1, 1, 20),   # Juan       → Sistema ERP        → 20 horas
    (10, 1, 40),  # Elena      → Sistema ERP        → 40 horas
    (2, 2, 30),   # María      → Campaña Digital    → 30 horas
    (6, 2, 60),   # Sofía      → Campaña Digital    → 60 horas
    (3, 3, 80),   # Carlos     → App Móvil          → 80 horas
    (7, 3, 40),   # Miguel     → App Móvil          → 40 horas
    (4, 4, 20),   # Ana        → Expansión Mercado  → 20 horas
    (8, 4, 50),   # Laura      → Expansión Mercado  → 50 horas
    (5, 5, 60),   # Luis       → Auditoría Anual    → 60 horas
    (9, 5, 70),   # Diego      → Auditoría Anual    → 70 horas
]
with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()
    cursor.executemany("""
        INSERT INTO empleados_proyectos (empleado_id, proyecto_id, horas_asignadas)
        VALUES (?, ?, ?)
    """, relaciones)

print(f"{len(relaciones)} relaciones insertadas correctamente.") #Nos informa la cantidad de relaciones insertadas

Tabla 'empleados_proyectos' creada correctamente.
10 relaciones insertadas correctamente.


### ***Ejercicio 8. Consultas analíticas***

**NOTA**: *Verán en este código que he empezado a relacionarme mas con los codigos para decorar los prints en consola(Ejemplo: print("  " + "-" * 57)). Brindando asi resultados mas agradables y legibles a la vista humana a la hora de ver los resultados del codigo ejecutado.*

In [15]:
with sqlite3.connect(DB_PATH) as conn:

    # Consulta 1: Total de horas asignadas por proyecto
    print("=== Total de horas por proyecto ===")
    filas = conn.execute("""
        SELECT p.nombre, SUM(ep.horas_asignadas) AS total_horas
        FROM empleados_proyectos ep
        JOIN proyectos p ON ep.proyecto_id = p.id
        GROUP BY p.id
    """).fetchall()
    for fila in filas:
        print(f"  {fila[0]:<25} {fila[1]:>5} horas")

    # Consulta 2: Empleado con mayor número de horas asignadas en total
    print("\n=== Empleado con más horas asignadas ===")
    fila = conn.execute("""
        SELECT e.nombre, SUM(ep.horas_asignadas) AS total_horas
        FROM empleados_proyectos ep
        JOIN empleados e ON ep.empleado_id = e.id
        GROUP BY e.id
        ORDER BY total_horas DESC
        LIMIT 1
    """).fetchone()
    print(f"  {fila[0]}: {fila[1]} horas")

    # Consulta 3: Departamentos con más de 2 proyectos activos
    print("\n=== Departamentos con más de 2 proyectos ===")
    filas = conn.execute("""
        SELECT d.nombre, COUNT(p.id) AS total_proyectos
        FROM departamentos d
        JOIN proyectos p ON p.departamento_id = d.id
        GROUP BY d.id
        HAVING COUNT(p.id) > 2
    """).fetchall()
    if filas:
        for fila in filas:
            print(f"  {fila[0]}: {fila[1]} proyectos")
    else:
        print("  Ningún departamento tiene más de 2 proyectos.")

    # Consulta 4: Gasto total por departamento (salarios + presupuesto de proyectos)
    print("\n=== Gasto total por departamento ===")
    filas = conn.execute("""
        SELECT 
            d.nombre,
            COALESCE(SUM(e.salario), 0)  AS gasto_salarios,
            COALESCE(SUM(p.presupuesto), 0) AS gasto_proyectos,
            COALESCE(SUM(e.salario), 0) + COALESCE(SUM(p.presupuesto), 0) AS gasto_total
        FROM departamentos d
        LEFT JOIN empleados e  ON e.departamento_id = d.id
        LEFT JOIN proyectos p  ON p.departamento_id = d.id
        GROUP BY d.id
    """).fetchall()
    print(f"\n  {'Departamento':<20} {'Salarios':>12} {'Proyectos':>12} {'Total':>12}")
    print("  " + "-" * 57)
    for fila in filas:
        print(f"  {fila[0]:<20} {fila[1]:>12,.0f} {fila[2]:>12,.0f} {fila[3]:>12,.0f}")

=== Total de horas por proyecto ===
  Sistema ERP                 180 horas
  Campaña Digital             270 horas
  App Móvil                   360 horas
  Expansión Mercado           210 horas
  Auditoría Anual             390 horas

=== Empleado con más horas asignadas ===
  Carlos García: 240 horas

=== Departamentos con más de 2 proyectos ===
  Ningún departamento tiene más de 2 proyectos.

=== Gasto total por departamento ===

  Departamento             Salarios    Proyectos        Total
  ---------------------------------------------------------
  Recursos Humanos          118,000       60,000      178,000
  Marketing                 105,000       40,000      145,000
  Desarrollo                143,500      100,000      243,500
  Ventas                     87,000       50,000      137,000
  Finanzas                  135,000       30,000      165,000
